# Silver Layer: Cleaning

**Target Tables:**
- **Read:** `bronze_ad_links` (Bronze Layer)
- **Write:** `silver_clean_ads` (Parquet)

**Objective:**
This notebook is responsible for cleaning and transforming the raw data extracted in the Bronze layer. It applies business rules, handles missing values, normalizes text (e.g., prices, descriptions, and dates), and structures the data into a clean, queryable format suitable for analysis in the Gold layer.

**To-Do:**
- Define strict schema enforcement for the Silver tables.
- Implement string manipulation for currency (BRL) and dates.
- Filter out invalid, duplicated, or outlier advertisements.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[2])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert, select
from sqlalchemy.orm import Session
import polars as pl
import re
import unicodedata

# Importing from 'app' module
from app.config import db_engine
from app.models import (
    GeneralSearch,
    InformationExtraction,
    SilverCleanAd
)

In [ ]:
with db_engine.connect() as connection:
    # Table with general instructions
    df_general = (
        pl.read_database(
            select(GeneralSearch)
            .where(
                (GeneralSearch.available == True)
                & (GeneralSearch.status == 200)
            ),
            connection=connection
        )
    )

    # Details' table
    df_details = (
        pl.read_database(
            select(InformationExtraction)
            .where(
                (InformationExtraction.price > 50)
                & (InformationExtraction.price < 10_000)
                & (InformationExtraction.title.isnot(None))
            ),
            connection=connection
        )
    )

In [ ]:
def clean_col_name(col_name: str) -> str:
    lower_name = col_name.lower()
    
    # Removes special characters
    no_special_chars_name = (
        unicodedata
        .normalize('NFKD', lower_name)
        .encode('ASCII', 'ignore')
        .decode('utf-8')
    )

    # Replace spaces by '_'
    no_spaces_name = re.sub(r'[ -]', '_', no_special_chars_name)

    # Removes duplicated underscores
    final_name = re.sub(r'_+', '_', no_spaces_name)

    return final_name

In [ ]:
initial_full_df = (
    df_general
    .join(
        df_details,
        left_on="id",
        right_on="general_search_id",
        how="right" # Discard invalid or out-of-region general scraps
    )
    .unnest('specifications')
)

# Renaming coluns by removing spaces and special characters, also lowering them
col_mapping = {col: clean_col_name(col) for col in initial_full_df.columns}
renamed_initial_full_df = initial_full_df.rename(col_mapping)

In [5]:
# Regex
needs_repair_regex = r'(?i)(defeito|detalhe|quebrado|peças|n[ãa]o liga|bateria viciada)'
urgent_sale_regex = r'(?i)(urgente|torro|dinheiro)'

# Final df
col_normalized_full_df = (
    renamed_initial_full_df
    # ET and business rules
    .with_columns(
        # --- Booleans ---
        (pl.col('para_doacao').str.to_lowercase().str.strip_chars() == 'sim').fill_null(False).alias('for_donation'),
        (pl.col('aceita_trocas').str.to_lowercase().str.strip_chars() == 'sim').fill_null(False).alias('accept_trades'),
        
        # --- Dates ---
        pl.col('datetime').dt.date().alias('date'),
        
        # --- Number Extraction ---
        pl.col('memoria_ram').fill_null("0").str.extract(r'(\d+)').cast(pl.Int32).alias('ram_gb'),
        pl.col('armazenamento').fill_null("0").str.extract(r'(\d+)').cast(pl.Int32).alias('storage_gb'),
        pl.col('tamanho_de_tela').fill_null("0").str.extract(r'(\d+)').cast(pl.Float32).alias('screen_size_pol'),
        
        # --- Lists and free texts ---
        pl.col('title').str.replace_all(r'\s+', ' ').str.strip_chars().str.to_titlecase().alias('title'),
        pl.col('description').str.replace_all(r'\s+', ' ').str.strip_chars().fill_null("Not Informed").str.to_titlecase().alias('description'),
        pl.col('caracteristicas').str.replace_all("Inclui ", "").str.split(r', ').fill_null(["Not Informed"]).alias('characteristics'),
        
        # --- Cleaning future categories ---
        pl.col('region').str.to_uppercase().alias('region'),
        pl.col('marca').fill_null("Not Informed").str.strip_chars().alias('brand'),
        pl.col('condicao').fill_null("Not Informed").str.strip_chars().alias('item_condition'),
        pl.col('marca_do_processador').fill_null("Not Informed").alias('cpu_brand'),
        pl.col('modelo_do_processador').fill_null("Not Informed").alias('cpu_model'),
        pl.col('marca_da_placa_de_video').fill_null("Not Informed").alias('gpu_brand'),
    )
    # Feature extraction
    .with_columns(
        (
            (pl.col('description').str.contains(needs_repair_regex))
            | (pl.col('title').str.contains(needs_repair_regex))
        ).alias('needs_repair'),
        (
            (pl.col('description').str.contains(urgent_sale_regex))
            | (pl.col('title').str.contains(urgent_sale_regex))
        ).alias('urgent_sale')
    )
    # Typing as categorical
    .with_columns(
        pl.col([
            'category',
            'subcategory',
            'region',
            'store',
            'currency',
            'brand',
            'item_condition',
            'cpu_brand',
            'cpu_model'
        ]).cast(pl.Categorical)
    )
)

In [6]:
# Conditions Map Priority
conditions_map_priority = {
    'defeito': 9999,
    'novo': 1,
    'excelente': 2,
    'bom': 3
}

# Feature extraction
unique_characteristics_list = (
    col_normalized_full_df
    .explode('characteristics')
    .get_column('characteristics')
    .str.to_titlecase()
    .drop_nulls()
    .unique()
    .sort()
    .to_list()
)

# Renaming coluns by removing spaces and special characters, also lowering them
feature_cols_mapping = {col: clean_col_name(col) for col in unique_characteristics_list}

# One-Hot Encoding
characteristics_expressions = [
    pl.col('characteristics')
    .list.contains(carac)
    .fill_null(False)
    .alias(f'has_{feature_cols_mapping[carac]}')
    for carac in unique_characteristics_list
]

characteristics_feature_extraction_df = (
    col_normalized_full_df
    .with_columns(characteristics_expressions)
    .with_columns(
        pl.col('item_condition').cast(pl.String).str.to_lowercase().alias('lowered_str_item_condition')
    )
    .with_columns(
        pl.when(pl.col('lowered_str_item_condition').str.contains('defeito')).then(9999)
        .when(pl.col('lowered_str_item_condition').str.contains('novo')).then(1)
        .when(pl.col('lowered_str_item_condition').str.contains('excelente')).then(2)
        .when(pl.col('lowered_str_item_condition').str.contains('bom')).then(3)
        .otherwise(0) # Fallback
        .cast(pl.Int16)
        .alias('item_condition_indicator')
    )
    .rename({
        'has_acessorios': 'has_accessories',
        'has_cabos': 'has_cables',
        'has_wi_fi': 'has_wifi'
    })
    # Final selection
    .select(
        'id',
        'date',
        'category',
        'subcategory',
        'item',
        'brand',
        'item_condition',
        'item_condition_indicator',
        'title',
        'description',
        'characteristics', 
        'price',
        'currency',
        'ram_gb',
        'storage_gb',
        'cpu_brand',
        'cpu_model',
        'gpu_brand',
        'screen_size_pol',
        'for_donation',
        'accept_trades',
        'region',
        'store',
        'url',
        'link',
        'first_image_src',
        'needs_repair',
        'urgent_sale',
        'has_accessories',
        'has_bluetooth',
        'has_cables',
        'has_hdmi',
        'has_ssd',
        'has_wifi',
        'ad_id'
    )
    # Dropping duplicates of ad id + date
    .unique(subset=['ad_id', 'date'], keep='any')
)

In [7]:
if not characteristics_feature_extraction_df.is_empty():
    with Session(db_engine) as session:
        # Saving on sqlite
        session.execute(insert(SilverCleanAd), characteristics_feature_extraction_df.to_dicts())
        session.commit()
else:
    print("No data found to insert.")